## Step 4: Baseline Model (MLP)

In this step, a baseline classification model is developed using a Multi-Layer Perceptron (MLP). The objective is to establish an initial performance benchmark for pneumonia detection using preprocessed chest X-ray images.

Since MLP models do not inherently capture spatial relationships in images, the input images are flattened into one-dimensional feature vectors. This simplification allows us to evaluate how well a basic model performs without leveraging spatial feature extraction.

A reduced image size and a subset of the dataset are used to ensure faster training and experimentation. The performance of this baseline model will serve as a reference point for comparing more advanced models such as Convolutional Neural Networks (CNNs) and transfer learning approaches in later stages.

This step provides insight into the limitations of simple models when applied to complex medical imaging tasks and highlights the necessity of more sophisticated architectures.


In [6]:
#  Import required libraries

import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [7]:
# Load dataset

df = pd.read_csv("/kaggle/input/datasets/sairasagnak/balanced-data-rsna/balanced_data.csv")
print("Dataset shape:", df.shape)

Dataset shape: (12024, 8)


In [8]:
#  Fix image paths (Kaggle)

DATA_DIR = "/kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset"
IMAGE_DIR = os.path.join(DATA_DIR, "Training", "Images")

df['image_path'] = df['patientId'].apply(lambda x: os.path.join(IMAGE_DIR, x + ".png"))

# keep only valid paths (safety)
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print("After path filtering:", df.shape)

After path filtering: (12024, 8)


In [9]:
#  Define preprocessing function

IMG_SIZE = 64  # smaller for MLP

def preprocess_image(path):
    img = cv2.imread(path, 0)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

In [10]:
#  Prepare data (use subset for speed)

df = df.sample(min(2000, len(df)), random_state=42)

images = []
labels = []

for _, row in df.iterrows():
    img = preprocess_image(row['image_path'])
    if img is None:
        continue
    images.append(img)
    labels.append(row['Target'])

X = np.array(images)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2000, 64, 64)
y shape: (2000,)


In [12]:
#  Flatten images for MLP

X = X.reshape(X.shape[0], -1)
print("Flattened shape:", X.shape)

Flattened shape: (2000, 4096)


In [15]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1600, 4096)
Test size: (400, 4096)


In [16]:
# Train MLP model

model = MLPClassifier(hidden_layer_sizes=(128,), max_iter=10, random_state=42)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(128,), max_iter=10, random_state=42)

In [17]:
#  Evaluate model

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.75
              precision    recall  f1-score   support

           0       0.81      0.71      0.75       216
           1       0.70      0.80      0.75       184

    accuracy                           0.75       400
   macro avg       0.75      0.75      0.75       400
weighted avg       0.76      0.75      0.75       400



## Step 4: Observations

* The baseline MLP model achieved an accuracy of 75% on the test set, indicating moderate performance on the pneumonia classification task.
* The model demonstrated slightly higher precision for the normal class (0.81) compared to the pneumonia class (0.70), suggesting a tendency to be more confident when predicting non-pneumonia cases.
* Recall for the pneumonia class (0.80) is relatively high, indicating that the model is able to identify most positive cases, which is crucial in medical diagnosis scenarios.
* Since the MLP model operates on flattened image inputs, it fails to capture spatial relationships and structural patterns present in chest X-ray images.
* The balanced dataset helped prevent bias toward any single class, resulting in comparable performance across both classes.

These results highlight the limitations of traditional fully connected networks for image-based tasks and motivate the need for convolutional architectures that can better capture spatial features.
